In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import DataLoader
import numpy as np
from PIL import Image
import json
import os

from datasets.rayan_dataset import RayanDataset
# from utils.dump_scores import save_scores
from utils.dump_scores import DumpScores

In [1]:
class ZeroShotAnomalyDetector:
    def __init__(self):
        # Load a pre-trained vision transformer as the backbone
        self.backbone = models.vit_b_16(pretrained=True)
        self.backbone.eval()
        
        # Remove the classification head
        self.backbone.heads = nn.Identity()
        
        # Feature pyramid for multi-scale analysis
        self.pyramid_layers = [2, 5, 8, 11]  # Layer indices for feature extraction
        
        # Normalization parameters
        self.mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        
    def preprocess_image(self, image):
        # Convert to tensor and normalize
        if isinstance(image, np.ndarray):
            image = torch.from_numpy(image).permute(2, 0, 1)
        image = image.float() / 255.0
        image = (image - self.mean) / self.std
        return image.unsqueeze(0)
    
    def extract_features(self, x):
        features = []
        # Get intermediate features from different transformer blocks
        for i, block in enumerate(self.backbone.encoder.layers):
            x = block(x)
            if i in self.pyramid_layers:
                features.append(x)
        return features
    
    def compute_patch_scores(self, features):
        # Compute local patch-wise anomaly scores
        patch_scores = []
        for feat in features:
            # Calculate mean feature vector
            mean_feat = feat.mean(dim=1, keepdim=True)
            
            # Compute distance from mean for each patch
            distances = torch.norm(feat - mean_feat, dim=-1)
            
            # Normalize distances
            patch_scores.append(distances)
            
        return torch.stack(patch_scores, dim=1)
    
    def interpolate_scores(self, scores, size=(224, 224)):
        # Interpolate patch scores to original image size
        return nn.functional.interpolate(
            scores, 
            size=size, 
            mode='bilinear', 
            align_corners=False
        )
    
    def detect(self, image):
        # Preprocess image
        x = self.preprocess_image(image)
        
        # Extract multi-scale features
        features = self.extract_features(x)
        
        # Compute patch-wise anomaly scores
        patch_scores = self.compute_patch_scores(features)
        
        # Generate pixel-level anomaly map
        pixel_scores = self.interpolate_scores(patch_scores.mean(dim=1, keepdim=True))
        pixel_scores = pixel_scores.squeeze().cpu().numpy()
        
        # Compute image-level score as the maximum patch score
        img_score = float(patch_scores.max().cpu().numpy())
        
        # Normalize scores to [0, 1]
        pixel_scores = (pixel_scores - pixel_scores.min()) / (pixel_scores.max() - pixel_scores.min())
        img_score = (img_score - patch_scores.min().cpu().numpy()) / (
            patch_scores.max().cpu().numpy() - patch_scores.min().cpu().numpy()
        )
        
        return img_score, pixel_scores

In [ ]:
# Initialize detector
detector = ZeroShotAnomalyDetector()

# Create output directory
os.makedirs('output_scores', exist_ok=True)

# Process each class in the dataset
data_root = 'data'
for class_name in os.listdir(data_root):
    class_path = os.path.join(data_root, class_name)
    if not os.path.isdir(class_path):
        continue
        
    # Create dataset and dataloader
    dataset = RayanDataset(class_path)
    dataloader = DataLoader(dataset, batch_size=1, shuffle=False)
    
    # Process each image
    for image_path, image in dataloader:
        # Get image name without extension
        image_name = os.path.splitext(os.path.basename(image_path[0]))[0]
        
        # Detect anomalies
        img_score, pixel_scores = detector.detect(image[0])
        
        # Save scores
        save_scores(
            class_name,
            image_name,
            img_score,
            pixel_scores
        )